# Multi-Objective Bayesian Optimization with BoTorch

## Breast Cancer Detection Hyperparameter Optimization

This notebook demonstrates how to use **BoTorch** with **qNEHVI acquisition** for efficient multi-objective hyperparameter optimization.

### Key Features
- **Sample Efficiency:** 200 evaluations vs 1000 for NSGA-III (80% reduction)
- **State-of-the-art:** qNEHVI (Quasi-Monte Carlo Noisy Expected Hypervolume Improvement)
- **Intelligent Exploration:** GP-based uncertainty quantification
- **Resumable:** Checkpoint every N iterations

### Problem Specification

**5 Hyperparameters:**
1. Learning rate: [1e-5, 1e-3] (log-scale)
2. Weight decay: [1e-6, 1e-2] (log-scale)
3. Dropout: [0.0, 0.5]
4. Augmentation strength: [0.0, 1.0]
5. Unfreeze fraction: [0.0, 1.0]

**4 Objectives (all minimization):**
1. -PR-AUC (maximize PR-AUC)
2. -AUROC (maximize AUROC)
3. Brier score (minimize)
4. Robustness degradation (minimize)

## 1. Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Add project to path
project_root = Path.cwd()
if 'breast_cancer_detection' not in sys.path:
    sys.path.insert(0, str(project_root))

# Import BoTorch modules
from breast_cancer_detection.src.evaluation_functions import (
    create_evaluation_function, 
    decode_hyperparameters
)
from breast_cancer_detection.src.botorch_mobo import (
    MultiObjectiveGPModel,
    qNEHVIAcquisition,
    compute_reference_point,
    initial_sobol_sampling
)
from breast_cancer_detection.src.botorch_utils import (
    HyperparameterTransform,
    BoTorchCheckpoint
)
from breast_cancer_detection.src.preprocessing import MammographyPreprocessor
from breast_cancer_detection.src.datasets import (
    VinDRMammoBinaryDataset,
    create_breast_level_splits
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration

Set paths and optimization parameters here.

In [ ]:
# ========================================
# CONFIGURATION
# ========================================

# Data paths (UPDATE THESE!)
DATA_ROOT = r"C:\path\to\vindr\images"  # UPDATE THIS
CSV_FILE = r"C:\path\to\vindr.csv"      # UPDATE THIS

# Optimization parameters
N_INITIAL = 10        # Initial Sobol samples (2× dimensionality)
N_ITERATIONS = 38     # BO iterations after initial sampling
BATCH_SIZE = 5        # Candidates per BO iteration
TOTAL_BUDGET = N_INITIAL + N_ITERATIONS * BATCH_SIZE  # = 200

# Training parameters
TRAIN_BATCH_SIZE = 4
MAX_EPOCHS = 50
PATIENCE = 10

# Acquisition parameters
ACQ_SAMPLES = 128
ACQ_RESTARTS = 20
REF_POINT_OFFSET = 0.1

# Checkpointing
CHECKPOINT_FREQ = 5
OUTPUT_DIR = Path("results/botorch_mobo")
RUN_ID = f"notebook_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Hardware
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

# Resume from checkpoint (set to None for fresh start)
RESUME_FROM = None  # e.g., "results/botorch_mobo/run_001/checkpoint_iter10.pt"

print("Configuration:")
print(f"  Total budget: {TOTAL_BUDGET} evaluations")
print(f"  Initial: {N_INITIAL}, BO iterations: {N_ITERATIONS}, Batch size: {BATCH_SIZE}")
print(f"  Device: {DEVICE}")
print(f"  Output: {OUTPUT_DIR / RUN_ID}")

## 3. Load Dataset

Load VinDr-Mammo with breast-level split.

In [ ]:
def setup_dataset(data_root, csv_file, seed=42):
    """Setup VinDr-Mammo dataset with breast-level split."""
    print("Loading VinDr-Mammo dataset...")
    
    preprocessor = MammographyPreprocessor(
        target_size=(720, 480),
        aspect_ratio=1.5
    )
    
    dataset = VinDRMammoBinaryDataset(
        images_root=data_root,
        csv_file=csv_file,
        preprocessor=preprocessor
    )
    
    print(f"Total samples: {len(dataset)}")
    
    # Class counts
    all_labels = [dataset[i][1].item() for i in range(len(dataset))]
    n_benign = sum([1 for l in all_labels if l == 0])
    n_malignant = sum([1 for l in all_labels if l == 1])
    
    print(f"  Benign: {n_benign}")
    print(f"  Malignant: {n_malignant}")
    print(f"  Ratio: {n_benign/n_malignant:.2f}:1")
    
    # Breast-level split
    print("\nCreating breast-level train/val split...")
    train_dataset, val_dataset = create_breast_level_splits(
        dataset=dataset,
        train_ratio=0.8,
        random_state=seed,
        stratify=True
    )
    
    print(f"  Train: {len(train_dataset)} samples")
    print(f"  Val: {len(val_dataset)} samples")
    
    return train_dataset, val_dataset, n_benign, n_malignant

# Load data
train_dataset, val_dataset, n_benign, n_malignant = setup_dataset(
    DATA_ROOT, CSV_FILE, SEED
)

pos_weight = n_benign / n_malignant
print(f"\nPos weight: {pos_weight:.3f}")

## 4. Initialize Evaluation Function

Create the evaluation function that trains CNN and returns objectives.

In [ ]:
# Set random seeds
torch.manual_seed(SEED)
np.random.seed(SEED)

# Create evaluation function
print("Creating evaluation function...")
evaluate_fn = create_evaluation_function(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    device=DEVICE,
    batch_size=TRAIN_BATCH_SIZE,
    num_workers=2,
    patience=PATIENCE,
    max_epochs=MAX_EPOCHS,
    pos_weight=pos_weight,
    random_seed=SEED
)

print("✓ Evaluation function ready")
print(f"  Each evaluation trains a ResNet152 CNN (~30 min on GPU)")

## 5. Initialize BoTorch Components

In [ ]:
# Setup transforms
transform = HyperparameterTransform()
bounds_linear = transform.bounds_linear

print("Hyperparameter bounds (linear space):")
print(f"  Learning rate (log10): {bounds_linear[0, 0]:.1f} to {bounds_linear[1, 0]:.1f}")
print(f"  Weight decay (log10):  {bounds_linear[0, 1]:.1f} to {bounds_linear[1, 1]:.1f}")
print(f"  Dropout:               {bounds_linear[0, 2]:.1f} to {bounds_linear[1, 2]:.1f}")
print(f"  Aug strength:          {bounds_linear[0, 3]:.1f} to {bounds_linear[1, 3]:.1f}")
print(f"  Unfreeze fraction:     {bounds_linear[0, 4]:.1f} to {bounds_linear[1, 4]:.1f}")

# Initialize GP model
gp_model = MultiObjectiveGPModel(
    n_objectives=4,
    n_vars=5,
    bounds=bounds_linear
)

print(f"\n✓ BoTorch components initialized")
print(f"  GP model: {gp_model.n_objectives} objectives, {gp_model.n_vars} variables")

## 6. Evaluation Helper Function

In [ ]:
def evaluate_batch(evaluate_fn, X_batch, transform):
    """
    Evaluate a batch of hyperparameter configurations.
    
    Args:
        evaluate_fn: Evaluation function
        X_batch: (batch_size, n_vars) tensor in linear space
        transform: HyperparameterTransform instance
        
    Returns:
        Y_batch: (batch_size, n_objs) tensor of objectives
        hyperparams_list: List of hyperparameter dicts
    """
    batch_size = X_batch.shape[0]
    Y_batch = []
    hyperparams_list = []
    
    for i in range(batch_size):
        x = X_batch[i].numpy()
        
        # Decode to real hyperparameters
        hyperparams = transform.to_real_hyperparams(x)
        
        print(f"\n  [{i+1}/{batch_size}] Evaluating:")
        print(f"    LR={hyperparams['learning_rate']:.6f}, WD={hyperparams['weight_decay']:.6f}")
        print(f"    Dropout={hyperparams['dropout']:.2f}, Aug={hyperparams['augmentation_strength']:.2f}")
        
        # Evaluate (expensive!)
        metrics = evaluate_fn(hyperparams)
        
        # Convert to minimization objectives
        objectives = [
            -metrics["pr_auc"],
            -metrics["auroc"],
            metrics["brier"],
            metrics["robustness_degradation"]
        ]
        
        Y_batch.append(objectives)
        hyperparams_list.append(hyperparams)
        
        print(f"    Results: PR-AUC={metrics['pr_auc']:.4f}, AUROC={metrics['auroc']:.4f}")
        print(f"             Brier={metrics['brier']:.4f}, Robust={metrics['robustness_degradation']:.4f}")
    
    Y_batch = torch.tensor(Y_batch, dtype=torch.float64)
    
    return Y_batch, hyperparams_list

## 7. Phase 1: Initial Sobol Sampling

Generate space-filling initial design with Sobol sequences.

In [ ]:
# Create output directory
output_dir = OUTPUT_DIR / RUN_ID
output_dir.mkdir(parents=True, exist_ok=True)

if RESUME_FROM is not None:
    print(f"Resuming from checkpoint: {RESUME_FROM}")
    gp_model, X_train, Y_train, start_iter, extras = \
        BoTorchCheckpoint.resume_from_checkpoint(RESUME_FROM, MultiObjectiveGPModel)
    all_hyperparams = extras.get('all_hyperparams', [])
    print(f"  Loaded {len(X_train)} evaluations")
    print(f"  Starting from iteration {start_iter}")
else:
    print("="*80)
    print("PHASE 1: INITIAL SOBOL SAMPLING")
    print("="*80)
    
    # Generate Sobol samples
    X_init = initial_sobol_sampling(
        n_vars=5,
        n_samples=N_INITIAL,
        bounds=bounds_linear
    )
    
    print(f"\nEvaluating {N_INITIAL} initial Sobol samples...")
    print("This will take approximately:", f"{N_INITIAL * 0.5:.1f} hours (30 min/eval)")
    
    # Evaluate initial samples
    Y_init, hyperparams_init = evaluate_batch(evaluate_fn, X_init, transform)
    
    # Initialize training data
    X_train = X_init
    Y_train = Y_init
    all_hyperparams = hyperparams_init
    start_iter = 0
    
    print(f"\n✓ Initial sampling complete")
    print(f"  Evaluations: {len(X_train)}")

# Save initial results
df = pd.DataFrame(all_hyperparams)
df['pr_auc'] = -Y_train[:, 0].numpy()
df['auroc'] = -Y_train[:, 1].numpy()
df['brier'] = Y_train[:, 2].numpy()
df['robustness'] = Y_train[:, 3].numpy()
df.to_csv(output_dir / 'evaluations.csv', index=False)

print(f"\nResults saved to: {output_dir / 'evaluations.csv'}")

## 8. Phase 2: Bayesian Optimization Loop

Iteratively:
1. Fit GP models on observed data
2. Optimize qNEHVI acquisition
3. Evaluate selected candidates
4. Update training data

In [ ]:
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

print("="*80)
print("PHASE 2: BAYESIAN OPTIMIZATION")
print("="*80)
print(f"\nRunning {N_ITERATIONS} iterations with batch size {BATCH_SIZE}")
print(f"Expected time: {N_ITERATIONS * BATCH_SIZE * 0.5:.1f} hours")

for iteration in range(start_iter, N_ITERATIONS):
    print(f"\n{'='*80}")
    print(f"BO Iteration {iteration+1}/{N_ITERATIONS}")
    print(f"{'='*80}")
    
    # Update GP model
    print("\n[1/4] Fitting GP models...")
    gp_model.update_data(X_train, Y_train)
    gp_model.fit_models()
    print("      ✓ GPs fitted")
    
    # Compute reference point
    ref_point = compute_reference_point(Y_train, offset=REF_POINT_OFFSET)
    print(f"\n[2/4] Reference point: {ref_point.numpy()}")
    
    # Create acquisition function
    print(f"\n[3/4] Optimizing qNEHVI acquisition...")
    acq = qNEHVIAcquisition(
        model=gp_model.models,
        ref_point=ref_point,
        bounds=bounds_linear,
        X_baseline=X_train
    )
    
    # Generate candidates
    X_next = acq.optimize(
        q=BATCH_SIZE,
        num_restarts=ACQ_RESTARTS,
        raw_samples=512
    )
    
    print(f"      ✓ Generated {len(X_next)} candidates")
    
    # Evaluate candidates
    print(f"\n[4/4] Evaluating {BATCH_SIZE} candidates...")
    Y_next, hyperparams_next = evaluate_batch(evaluate_fn, X_next, transform)
    
    # Update training data
    X_train = torch.cat([X_train, X_next], dim=0)
    Y_train = torch.cat([Y_train, Y_next], dim=0)
    all_hyperparams.extend(hyperparams_next)
    
    # Statistics
    print(f"\nIteration {iteration+1} complete:")
    print(f"  Total evaluations: {len(X_train)}/{TOTAL_BUDGET}")
    
    # Find current Pareto front
    nds = NonDominatedSorting()
    fronts = nds.do(Y_train.numpy(), only_non_dominated_front=True)
    print(f"  Pareto front size: {len(fronts[0])}")
    
    # Save results
    df = pd.DataFrame(all_hyperparams)
    df['pr_auc'] = -Y_train[:, 0].numpy()
    df['auroc'] = -Y_train[:, 1].numpy()
    df['brier'] = Y_train[:, 2].numpy()
    df['robustness'] = Y_train[:, 3].numpy()
    df.to_csv(output_dir / 'evaluations.csv', index=False)
    
    # Checkpoint
    if (iteration + 1) % CHECKPOINT_FREQ == 0:
        checkpoint_path = output_dir / f'checkpoint_iter{iteration+1}.pt'
        BoTorchCheckpoint.save(
            checkpoint_path,
            gp_model,
            X_train,
            Y_train,
            iteration+1,
            all_hyperparams=all_hyperparams,
            config={
                'n_initial': N_INITIAL,
                'n_iterations': N_ITERATIONS,
                'batch_size': BATCH_SIZE
            }
        )
        print(f"  ✓ Checkpoint saved: {checkpoint_path.name}")

print("\n" + "="*80)
print("OPTIMIZATION COMPLETE")
print("="*80)

## 9. Extract Pareto Front

In [ ]:
# Compute final Pareto front
nds = NonDominatedSorting()
fronts = nds.do(Y_train.numpy(), only_non_dominated_front=True)
pareto_idx = fronts[0]

X_pareto = X_train[pareto_idx]
Y_pareto = Y_train[pareto_idx]

print(f"Final Statistics:")
print(f"  Total evaluations: {len(X_train)}")
print(f"  Pareto solutions: {len(pareto_idx)}")

# Save final results
final_results = {
    'X_all': X_train,
    'Y_all': Y_train,
    'X_pareto': X_pareto,
    'Y_pareto': Y_pareto,
    'pareto_indices': pareto_idx,
    'all_hyperparams': all_hyperparams,
    'gp_model': gp_model,
    'config': {
        'n_initial': N_INITIAL,
        'n_iterations': N_ITERATIONS,
        'batch_size': BATCH_SIZE,
        'total_budget': TOTAL_BUDGET
    }
}

results_path = output_dir / 'final_results.pkl'
with open(results_path, 'wb') as f:
    pickle.dump(final_results, f)

print(f"\n✓ Results saved to: {results_path}")

## 10. Display Pareto Front Solutions

In [ ]:
print("\nPareto Front Solutions:")
print("="*90)
print(f"{'ID':<5} {'PR-AUC':>8} {'AUROC':>8} {'Brier':>8} {'Robust':>8} {'LR':>10} {'WD':>10}")
print("-"*90)

for i, (idx, y) in enumerate(zip(pareto_idx, Y_pareto)):
    pr_auc = -y[0].item()
    auroc = -y[1].item()
    brier = y[2].item()
    robust = y[3].item()
    
    hp = all_hyperparams[idx]
    lr = hp['learning_rate']
    wd = hp['weight_decay']
    
    print(f"{i:<5} {pr_auc:>8.4f} {auroc:>8.4f} {brier:>8.4f} {robust:>8.4f} {lr:>10.6f} {wd:>10.6f}")

print("="*90)

## 11. Visualization: Convergence

In [ ]:
# Load evaluation history
df = pd.read_csv(output_dir / 'evaluations.csv')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PR-AUC
axes[0, 0].plot(df['pr_auc'], 'o-', alpha=0.6, label='Evaluations')
axes[0, 0].plot(df['pr_auc'].cummax(), 'r-', linewidth=2, label='Best so far')
axes[0, 0].set_xlabel('Evaluation')
axes[0, 0].set_ylabel('PR-AUC')
axes[0, 0].set_title('PR-AUC Convergence')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# AUROC
axes[0, 1].plot(df['auroc'], 'o-', alpha=0.6, label='Evaluations')
axes[0, 1].plot(df['auroc'].cummax(), 'r-', linewidth=2, label='Best so far')
axes[0, 1].set_xlabel('Evaluation')
axes[0, 1].set_ylabel('AUROC')
axes[0, 1].set_title('AUROC Convergence')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Brier Score
axes[1, 0].plot(df['brier'], 'o-', alpha=0.6, label='Evaluations')
axes[1, 0].plot(df['brier'].cummin(), 'r-', linewidth=2, label='Best so far')
axes[1, 0].set_xlabel('Evaluation')
axes[1, 0].set_ylabel('Brier Score')
axes[1, 0].set_title('Brier Score Convergence (lower is better)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Robustness Degradation
axes[1, 1].plot(df['robustness'], 'o-', alpha=0.6, label='Evaluations')
axes[1, 1].plot(df['robustness'].cummin(), 'r-', linewidth=2, label='Best so far')
axes[1, 1].set_xlabel('Evaluation')
axes[1, 1].set_ylabel('Robustness Degradation')
axes[1, 1].set_title('Robustness Degradation (lower is better)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Convergence plot saved to: {output_dir / 'convergence.png'}")

## 12. Visualization: Pareto Front (2D Projections)

In [ ]:
# Extract objectives
pr_auc_all = -Y_train[:, 0].numpy()
auroc_all = -Y_train[:, 1].numpy()
brier_all = Y_train[:, 2].numpy()
robust_all = Y_train[:, 3].numpy()

pr_auc_pareto = -Y_pareto[:, 0].numpy()
auroc_pareto = -Y_pareto[:, 1].numpy()
brier_pareto = Y_pareto[:, 2].numpy()
robust_pareto = Y_pareto[:, 3].numpy()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# PR-AUC vs AUROC
axes[0, 0].scatter(pr_auc_all, auroc_all, alpha=0.3, s=30, label='All evaluations')
axes[0, 0].scatter(pr_auc_pareto, auroc_pareto, c='red', s=100, marker='*', 
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[0, 0].set_xlabel('PR-AUC')
axes[0, 0].set_ylabel('AUROC')
axes[0, 0].set_title('PR-AUC vs AUROC')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# PR-AUC vs Brier
axes[0, 1].scatter(pr_auc_all, brier_all, alpha=0.3, s=30, label='All evaluations')
axes[0, 1].scatter(pr_auc_pareto, brier_pareto, c='red', s=100, marker='*',
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[0, 1].set_xlabel('PR-AUC')
axes[0, 1].set_ylabel('Brier Score')
axes[0, 1].set_title('PR-AUC vs Brier')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# PR-AUC vs Robustness
axes[0, 2].scatter(pr_auc_all, robust_all, alpha=0.3, s=30, label='All evaluations')
axes[0, 2].scatter(pr_auc_pareto, robust_pareto, c='red', s=100, marker='*',
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[0, 2].set_xlabel('PR-AUC')
axes[0, 2].set_ylabel('Robustness Degradation')
axes[0, 2].set_title('PR-AUC vs Robustness')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# AUROC vs Brier
axes[1, 0].scatter(auroc_all, brier_all, alpha=0.3, s=30, label='All evaluations')
axes[1, 0].scatter(auroc_pareto, brier_pareto, c='red', s=100, marker='*',
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[1, 0].set_xlabel('AUROC')
axes[1, 0].set_ylabel('Brier Score')
axes[1, 0].set_title('AUROC vs Brier')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# AUROC vs Robustness
axes[1, 1].scatter(auroc_all, robust_all, alpha=0.3, s=30, label='All evaluations')
axes[1, 1].scatter(auroc_pareto, robust_pareto, c='red', s=100, marker='*',
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[1, 1].set_xlabel('AUROC')
axes[1, 1].set_ylabel('Robustness Degradation')
axes[1, 1].set_title('AUROC vs Robustness')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Brier vs Robustness
axes[1, 2].scatter(brier_all, robust_all, alpha=0.3, s=30, label='All evaluations')
axes[1, 2].scatter(brier_pareto, robust_pareto, c='red', s=100, marker='*',
                   edgecolors='black', linewidths=1.5, label='Pareto front', zorder=10)
axes[1, 2].set_xlabel('Brier Score')
axes[1, 2].set_ylabel('Robustness Degradation')
axes[1, 2].set_title('Brier vs Robustness')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'pareto_front_2d.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Pareto front visualization saved to: {output_dir / 'pareto_front_2d.png'}")

## 13. Summary Statistics

In [ ]:
print("\n" + "="*80)
print("OPTIMIZATION SUMMARY")
print("="*80)

print(f"\nTotal evaluations: {len(X_train)}")
print(f"Pareto front size: {len(pareto_idx)}")

print(f"\nBest single-objective solutions:")
print(f"  Best PR-AUC:    {pr_auc_all.max():.4f}")
print(f"  Best AUROC:     {auroc_all.max():.4f}")
print(f"  Best Brier:     {brier_all.min():.4f}")
print(f"  Best Robustness: {robust_all.min():.4f}")

print(f"\nPareto front statistics:")
print(f"  PR-AUC range:    [{pr_auc_pareto.min():.4f}, {pr_auc_pareto.max():.4f}]")
print(f"  AUROC range:     [{auroc_pareto.min():.4f}, {auroc_pareto.max():.4f}]")
print(f"  Brier range:     [{brier_pareto.min():.4f}, {brier_pareto.max():.4f}]")
print(f"  Robustness range: [{robust_pareto.min():.4f}, {robust_pareto.max():.4f}]")

print(f"\nOutput files:")
print(f"  Evaluations CSV: {output_dir / 'evaluations.csv'}")
print(f"  Final results:   {output_dir / 'final_results.pkl'}")
print(f"  Convergence plot: {output_dir / 'convergence.png'}")
print(f"  Pareto plot:     {output_dir / 'pareto_front_2d.png'}")

## 14. Export Pareto Solutions for Further Analysis

In [ ]:
# Create DataFrame with Pareto solutions
pareto_solutions = []

for i, (idx, y) in enumerate(zip(pareto_idx, Y_pareto)):
    hp = all_hyperparams[idx]
    
    solution = {
        'solution_id': i,
        'learning_rate': hp['learning_rate'],
        'weight_decay': hp['weight_decay'],
        'dropout': hp['dropout'],
        'augmentation_strength': hp['augmentation_strength'],
        'unfreeze_fraction': hp['unfreeze_fraction'],
        'pr_auc': -y[0].item(),
        'auroc': -y[1].item(),
        'brier': y[2].item(),
        'robustness_degradation': y[3].item()
    }
    pareto_solutions.append(solution)

pareto_df = pd.DataFrame(pareto_solutions)
pareto_df.to_csv(output_dir / 'pareto_solutions.csv', index=False)

print("\nPareto Solutions:")
print(pareto_df.to_string(index=False))
print(f"\n✓ Pareto solutions saved to: {output_dir / 'pareto_solutions.csv'}")

## 15. Select Best Solution (by user preference)

Choose a solution from the Pareto front based on your preference.

In [ ]:
# Example: Select solution with best PR-AUC
best_pr_auc_idx = pr_auc_pareto.argmax()
best_solution = pareto_solutions[best_pr_auc_idx]

print("\nRecommended Solution (Best PR-AUC):")
print("="*50)
for key, value in best_solution.items():
    if key == 'solution_id':
        print(f"{key}: {value}")
    elif key in ['learning_rate', 'weight_decay']:
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value:.4f}")

print("\nUse these hyperparameters for training your final model!")

## Next Steps

1. **Analyze Pareto front** - Choose solution based on your preference (trade-off between objectives)
2. **Zero-shot evaluation** - Evaluate selected solution(s) on INbreast dataset
3. **Train final model** - Use selected hyperparameters for full training
4. **Compare with baselines** - If you have NSGA-III results, compare Pareto fronts

## Notes

- **Resuming:** Set `RESUME_FROM` variable to checkpoint path
- **Shorter runs:** Reduce `N_ITERATIONS` and `BATCH_SIZE` for testing
- **Parallel evaluation:** Current implementation is sequential; can be parallelized with multi-GPU
- **GP refitting:** GPs are refitted every iteration for maximum accuracy